In [1]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [2]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

1.2.2
1.5.5


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [4]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [5]:
def modify_date(date):
    return date[:10]

In [6]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [7]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [8]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [9]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [10]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [11]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [12]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P500"
end_swan = "Swan9"         #equals the last swan before the swan we are going to predict
start_swan="Swan5"
start = dt(2020,11,1)
end = dt(2022,1,3)
selected_stock_number = 19     #number of stock preselected
start_swan_date = dt(2022,1,3)
end_swan_date = dt(2023,1,1)
w1 = 0.1
w2 = 0.1
MA_days = 30
N = 17 # This parameter is set to help classification method like K-means

In [25]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("/content/BetaMartix S&P500 Dennis 2024.xlsx",index_col=0)

In [26]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [27]:
portfolio_beta= portfolio_beta.loc[:,"Swan1":end_swan]
portfolio_beta

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9
Ticker,,,,,,,,,
A,NaN,NaN,NaN,2.129657,1.408493,2.084547,1.194656,0.682441,0.763786
AAL,NaN,NaN,NaN,NaN,1.607495,1.783416,0.613406,1.207817,1.892777
AAPL,0.803017,1.957671,0.810491,1.374259,0.838225,0.265718,1.771183,1.901494,0.909584
ABBV,NaN,NaN,NaN,NaN,NaN,NaN,1.196566,0.593982,0.933795
ABC,NaN,NaN,1.799041,-0.098415,0.500850,0.644569,1.249513,1.141485,0.660946
...,...,...,...,...,...,...,...,...,...
YUM,NaN,NaN,-0.250673,-0.054483,0.745233,0.626758,1.146095,0.138012,1.372255
ZBH,NaN,NaN,NaN,-0.296565,1.240598,1.076392,0.777202,1.295412,1.468092
ZBRA,NaN,NaN,1.096156,0.459355,1.041248,1.660871,2.720121,0.815003,0.860525


In [24]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [ ]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9
Swan1,1.0,0.234634,0.200861,0.080872,-0.010629,0.155286,0.054266,0.227031,0.106071
Swan2,NaN,1.000000,0.372235,0.212392,0.260321,0.243822,-0.010243,0.216744,0.003824
Swan3,NaN,NaN,1.000000,0.100094,0.160429,0.324700,0.226882,0.353315,0.092120
Swan4,NaN,NaN,NaN,1.000000,0.033906,0.174311,0.079488,0.150590,0.005533
Swan5,NaN,NaN,NaN,NaN,1.000000,0.493237,0.167631,0.275134,0.443963
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.272880,0.504682,0.389450
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.250702,0.153897
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.299758
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [ ]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [ ]:
portfolio_beta

,Swan-4,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
stickers,,,,,,,,,,,,,,,
A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.129658,1.408494,2.084543,1.194656,0.682441,0.763786,1.349095,1.349100
AAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.607496,1.783414,0.613406,1.207817,1.892777,2.172210,2.172218
AAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.525701,0.207075,-0.048495,-0.125923,0.578707,1.299581,1.188325,1.188330
AAPL,NaN,NaN,1.977230,-0.37653,0.803019,1.957667,0.810489,1.374260,0.838224,0.265716,1.771181,1.901494,0.909584,1.040690,1.040694
ABBV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.196568,0.593981,0.933795,-0.760908,-0.760912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YUM,NaN,NaN,NaN,NaN,NaN,NaN,-0.250675,-0.054484,0.745234,0.626755,1.146096,0.138012,1.372254,1.181589,1.181593
ZBH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.296564,1.240597,1.076391,0.777204,1.295413,1.468093,0.596361,0.596363
ZBRA,NaN,NaN,NaN,NaN,NaN,NaN,1.096154,0.459356,1.041249,1.660868,2.720119,0.815003,0.860525,2.662073,2.662082


In [ ]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for random in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=random,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[random]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,40):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(50,len(sorted_count))):
      if sorted_count[p][1]>=1:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    # selected_beta['avg']=selected_beta['avg'].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

['DLTR', 'GIS', 'WMT', 'GILD', 'MNST', 'MKTX', 'CPB', 'REGN', 'AMGN', 'PG', 'NEM', 'CL', 'MRK', 'LLY', 'KMB', 'KR', 'DPZ', 'CLX', 'VZ', 'ORLY', 'NEM', 'RSG', 'AMT', 'SBUX', 'NEE', 'HSY', 'MKC', 'DTE', 'PSA', 'O', 'CNP', 'FE', 'MKTX', 'DLTR', 'SBAC', 'AWK', 'MCD', 'AZO', 'HRL', 'PPL', 'CMS', 'KO', 'ATO', 'ICE', 'XEL', 'ES', 'CHD', 'AAP', 'PNW', 'EXC', 'CCI', 'PEAK', 'WEC', 'PEG', 'SO', 'D', 'DUK', 'SRE', 'WELL', 'AEP', 'CINF', 'EVRG', 'ETR', 'MDLZ', 'PEP', 'LNT', 'NI', 'AEE', 'ED', 'ES', 'FE', 'MCD', 'SO', 'EXC', 'ORLY', 'EVRG', 'ETR', 'AWK', 'AZO', 'MDLZ', 'SBAC', 'ED', 'PNW', 'MKC', 'CLX', 'WEC', 'DUK', 'DTE', 'DLTR', 'VZ', 'NI', 'CCI', 'D', 'NEM', 'NEE', 'CHD', 'MKTX', 'AMT', 'ATO', 'AEE', 'HRL', 'ICE', 'AEP', 'CMS', 'CPB', 'KMB', 'KO', 'LNT', 'SRE', 'HSY', 'PPL', 'AAP', 'XEL', 'PEG', 'PEP', 'DUK', 'MCD', 'SBAC', 'MDLZ', 'SO', 'ETR', 'XEL', 'AMT', 'EVRG', 'EXC', 'D', 'ORLY', 'PNW', 'AZO', 'MKC', 'WEC', 'PPL', 'ATO', 'FE', 'CHD', 'NEE', 'SRE', 'ES', 'ICE', 'PEG', 'AAP', 'WM', 'ED', 'C

In [ ]:
ct

array([[ 0.82311856,  0.58831482,  0.97881998,  0.50156887,  0.78398758],
       [ 1.5761811 ,  2.37567765,  1.6191697 ,  1.45117568,  1.43137724],
       [ 1.01125011,  1.32971637,  0.79006   ,  1.08251784,  0.80042428],
       [ 1.34395054,  0.89375745,  0.80574194,  1.00395245,  1.57329361],
       [ 1.62840535,  2.02860979,  0.37186388,  0.71610996,  1.37800633],
       [ 1.15120899,  0.90481416,  0.58181678,  0.03008322,  1.25360701],
       [ 0.7985181 ,  0.38236479,  0.50108004,  1.24984179,  0.68574542],
       [ 0.64752718,  0.6853599 ,  1.77695548,  1.28910494,  1.09074067],
       [ 1.28688361,  1.5267788 ,  1.258259  ,  1.05934866,  1.21405049],
       [ 0.57564548,  0.14157641,  0.28300635, -0.01249442,  0.87555477],
       [ 0.61290929,  2.05497814,  0.23854383,  1.81440495,  0.91505371],
       [ 1.05184745,  2.0448101 ,  1.14664414,  2.24751236,  1.95151203],
       [ 1.70903541,  2.25591019, -2.5332745 ,  1.77515459,  0.62814325],
       [ 0.43910653,  0.87733728,  0.7

In [ ]:
res

array([0.73516196, 1.69071628, 1.00279372, 1.1241392 , 1.22459906,
       0.78430603, 0.72351003, 1.09793764, 1.26906411, 0.37265772,
       1.12717798, 1.68846522, 0.76699379, 0.80140574, 1.50608993,
       0.78956669, 2.15083815])

In [ ]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, AAP to XEL
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan5   50 non-null     float64
 1   Swan6   50 non-null     float64
 2   Swan7   50 non-null     float64
 3   Swan8   50 non-null     float64
 4   Swan9   50 non-null     float64
 5   Group   50 non-null     int32  
 6   avg     50 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 2.9+ KB


In [ ]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [ ]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg,PR
stickers,,,,,,,,
AAP,0.207075,-0.048495,-0.125923,0.578707,1.299581,9,0.382189,0.617811
AEE,1.139264,-0.008178,-0.139507,0.001824,0.833510,9,0.365382,0.634618
AEP,0.764573,0.012211,0.305260,-0.296070,0.959958,9,0.349186,0.650814
AMT,0.569595,0.048213,0.653109,-0.349966,0.824960,9,0.349182,0.650818
ATO,0.548668,0.468099,-0.135669,0.164407,0.959928,9,0.401086,0.598914
AWK,0.315968,-0.006396,-0.054045,0.054179,0.890179,9,0.239977,0.760023
AZO,-0.451106,-0.302805,-0.147542,-0.403255,0.955177,9,-0.069906,1.069906
CCI,1.078170,0.276796,0.251913,0.188927,0.854298,9,0.530021,0.469979
CHD,0.597611,-0.086318,0.238837,-0.333092,0.583540,9,0.200115,0.799885


# Step 2 - Download data and adjuste price based on Swan Beta

In [ ]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [ ]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['AAP' 'AEE' 'AEP' 'AMT' 'ATO' 'AWK' 'AZO' 'CCI' 'CHD' 'CLX' 'CMS' 'CNP'
 'CPB' 'D' 'DLTR' 'DTE' 'DUK' 'ED' 'ES' 'ETR' 'EVRG' 'EXC' 'FE' 'HRL'
 'HSY' 'ICE' 'KMB' 'KO' 'LNT' 'MCD' 'MDLZ' 'MKC' 'MKTX' 'NEE' 'NEM' 'NI'
 'O' 'ORLY' 'PEG' 'PEP' 'PNW' 'PPL' 'SBAC' 'SJM' 'SO' 'SRE' 'VZ' 'WEC'
 'WM' 'XEL']


In [ ]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  50 of 50 completed


In [ ]:
yahooDataFrame

Ticker,AAP,AEE,AEP,AMT,ATO,AWK,AZO,CCI,CHD,CLX,...,PNW,PPL,SBAC,SJM,SO,SRE,VZ,WEC,WM,XEL
Date,,,,,,,,,,,,,,,,,,,,,
2020-11-02,139.088150,74.613701,79.447807,210.895905,86.600853,144.427719,1141.150024,135.812775,84.117851,193.531815,...,69.026711,23.973215,281.357330,104.086525,50.797901,56.540134,46.420441,91.307068,104.056244,63.899658
2020-11-03,144.646881,75.609062,79.151711,211.564301,88.523102,147.048599,1164.650024,136.553650,85.170517,191.127533,...,69.614220,24.595781,278.951416,104.372040,51.529613,57.195824,46.460670,92.289726,107.854820,64.967926
2020-11-04,142.901154,74.828918,78.507286,214.129486,86.673393,145.466721,1154.020020,138.069489,85.907387,191.748871,...,69.988846,23.811178,289.505188,102.230698,51.443520,56.898983,46.034283,90.129669,109.114731,64.736473
2020-11-05,143.728088,73.950157,78.994965,216.703674,86.156570,148.087570,1149.510010,138.742264,86.156204,191.784882,...,71.930145,23.930574,293.147675,103.319221,51.796471,56.269878,46.782478,89.943764,112.537193,65.635582
2020-11-06,143.682190,73.394218,79.177826,218.717911,84.905289,150.820724,1158.530029,139.721588,86.605972,196.377335,...,71.342644,23.546799,293.914520,104.050842,51.615688,55.304066,47.088181,90.616577,112.151703,65.261703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,221.037323,81.150360,79.200851,264.678223,97.510857,174.904892,2047.060059,181.148834,97.446419,157.713242,...,61.268475,27.055386,370.892822,123.967476,60.677647,60.132294,44.302700,87.423470,157.233505,61.562050
2021-12-28,222.560730,81.956741,80.228607,267.206726,96.920502,177.343887,2059.939941,182.520126,97.271622,158.673050,...,62.006645,27.275274,372.218292,125.217789,61.375416,60.423336,44.336346,88.542877,158.529480,62.299099
2021-12-29,225.271118,82.410942,80.565117,267.567993,98.232391,178.534836,2104.040039,183.404877,98.048401,158.165482,...,62.140049,27.293596,372.431213,125.125198,61.683514,60.728252,44.033588,88.928253,159.719849,62.612343


In [ ]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [ ]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [ ]:
market_drop.to_csv("market_drop.csv")

In [ ]:
market_drop

,0
0,0.0000
1,0.0045
2,0.0089
3,0.0134
4,0.0179
...,...
289,0.0000
290,0.0000
291,0.0000
292,0.0000


In [ ]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [ ]:
yahooDataFrame


Ticker,AAP,AEE,AEP,AMT,ATO,AWK,AZO,CCI,CHD,CLX,...,PNW,PPL,SBAC,SJM,SO,SRE,VZ,WEC,WM,XEL
Date,,,,,,,,,,,,,,,,,,,,,
2020-11-02,139.088150,74.613701,79.447807,210.895905,86.600853,144.427719,1141.150024,135.812775,84.117851,193.531815,...,69.026711,23.973215,281.357330,104.086525,50.797901,56.540134,46.420441,91.307068,104.056244,63.899658
2020-11-03,144.398110,75.484744,79.027336,211.231865,88.363327,146.889802,1165.016398,136.227957,85.093819,190.878493,...,69.502222,24.540649,278.417698,104.176700,51.462320,57.047636,46.391524,92.181790,107.614212,64.894127
2020-11-04,142.415078,74.585582,78.263304,213.464031,86.363998,145.156033,1154.738011,137.418189,85.754383,191.254725,...,69.766148,23.705618,288.409676,101.852284,51.310650,56.607421,45.898781,89.921191,108.633302,64.591035
2020-11-05,142.992009,73.588088,78.625340,215.689709,85.693517,147.611366,1150.586807,137.756878,85.925172,191.040746,...,71.585547,23.770844,291.477500,102.743409,51.595047,55.835752,46.575149,89.630523,111.789610,65.413566
2020-11-06,142.699234,72.914195,78.682930,217.350845,84.295716,150.172860,1159.979724,138.395998,86.295744,195.359500,...,70.886082,23.336849,291.677630,103.276213,51.347561,54.734104,46.809416,90.195013,111.156487,64.966819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,221.037323,81.150360,79.200851,264.678223,97.510857,174.904892,2047.060059,181.148834,97.446419,157.713242,...,61.268475,27.055386,370.892822,123.967476,60.677647,60.132294,44.302700,87.423470,157.233505,61.562050
2021-12-28,222.560730,81.956741,80.228607,267.206726,96.920502,177.343887,2059.939941,182.520126,97.271622,158.673050,...,62.006645,27.275274,372.218292,125.217789,61.375416,60.423336,44.336346,88.542877,158.529480,62.299099
2021-12-29,225.271118,82.410942,80.565117,267.567993,98.232391,178.534836,2104.040039,183.404877,98.048401,158.165482,...,62.140049,27.293596,372.431213,125.125198,61.683514,60.728252,44.033588,88.928253,159.719849,62.612343


In [ ]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['AAP', 'AEE', 'AEP', 'AMT', 'ATO', 'AWK', 'AZO', 'CCI', 'CHD',
       'CLX', 'CMS', 'CNP', 'CPB', 'D', 'DLTR', 'DTE', 'DUK', 'ED', 'ES',
       'ETR', 'EVRG', 'EXC', 'FE', 'HRL', 'HSY', 'ICE', 'KMB', 'KO',
       'LNT', 'MCD', 'MDLZ', 'MKC', 'MKTX', 'NEE', 'NEM', 'NI', 'O',
       'ORLY', 'PEG', 'PEP', 'PNW', 'PPL', 'SBAC', 'SJM', 'SO', 'SRE',
       'VZ', 'WEC', 'WM', 'XEL'], dtype=object)

# Step3 - MVO

In [ ]:
#Set the paramter for MVO process
Portfolio_Date=end

In [ ]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [ ]:
#calculated rf
bond_yield = quandl.get('FRED/DGS3MO')
bond_yield=np.log(bond_yield/100+1)
date=list(map(str,bond_yield.index.values))
date=list(map(modify_date,date))
week=pd.to_datetime(date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
rf_week=bond_yield.iloc[week_difference-1]

In [ ]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [ ]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [ ]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [ ]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [ ]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


In [ ]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [ ]:
# Read OPT-data
optdata = pd.read_excel("optdata_SP500_20220103.xlsx",index_col=0,sheet_name = 0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [ ]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [ ]:
preselected_stock

,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
stickers,,,,,,,,,,,,
AAP,0.207075,-0.048495,-0.125923,0.578707,1.299581,9,0.382189,0.617811,0.206087,0.262889,0.783932,0.547009
AEE,1.139264,-0.008178,-0.139507,0.001824,0.833510,9,0.365382,0.634618,0.145098,0.169984,0.853600,0.586326
AEP,0.764573,0.012211,0.305260,-0.296070,0.959958,9,0.349186,0.650814,0.177847,0.165063,1.077449,0.675546
AMT,0.569595,0.048213,0.653109,-0.349966,0.824960,9,0.349182,0.650818,0.206995,0.215074,0.962434,0.638476
ATO,0.548668,0.468099,-0.135669,0.164407,0.959928,9,0.401086,0.598914,0.148051,0.212828,0.695638,0.499524
AWK,0.315968,-0.006396,-0.054045,0.054179,0.890179,9,0.239977,0.760023,0.150622,0.199392,0.755403,0.660566
AZO,-0.451106,-0.302805,-0.147542,-0.403255,0.955177,9,-0.069906,1.069906,0.366687,0.263963,1.389165,1.261022
CCI,1.078170,0.276796,0.251913,0.188927,0.854298,9,0.530021,0.469979,0.180552,0.212659,0.849020,0.433050
CHD,0.597611,-0.086318,0.238837,-0.333092,0.583540,9,0.200115,0.799885,0.143704,0.135848,1.057825,0.822686


In [ ]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


# Optimization

In [ ]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [ ]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('AAP', 0.0), ('AEE', 0.0), ('AEP', 0.0), ('AMT', 0.0), ('ATO', 0.0), ('AWK', 0.0), ('AZO', 0.05), ('CCI', 0.0), ('CHD', 0.05), ('CLX', 0.0), ('CMS', 0.0), ('CNP', 0.0), ('CPB', 0.0), ('D', 0.0), ('DLTR', 0.0), ('DTE', 0.05), ('DUK', 0.05), ('ED', 0.0), ('ES', 0.0), ('ETR', 0.0), ('EVRG', 0.05), ('EXC', 0.05), ('FE', 0.0), ('HRL', 0.0), ('HSY', 0.04084), ('ICE', 0.05), ('KMB', 0.00916), ('KO', 0.0), ('LNT', 0.0), ('MCD', 0.05), ('MDLZ', 0.05), ('MKC', 0.0), ('MKTX', 0.0), ('NEE', 0.05), ('NEM', 0.0), ('NI', 0.05), ('O', 0.05), ('ORLY', 0.05), ('PEG', 0.05), ('PEP', 0.0), ('PNW', 0.0), ('PPL', 0.05), ('SBAC', 0.0), ('SJM', 0.0), ('SO', 0.05), ('SRE', 0.05), ('VZ', 0.05), ('WEC', 0.0), ('WM', 0.05), ('XEL', 0.0)])
Expected annual return: 2.6%
Annual volatility: 10.9%
Sharpe Ratio: 0.14


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.025748056141200692, 0.10884666394021796, 0.14468110983952642)

In [ ]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [ ]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.14468110983927085


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [ ]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.01]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [ ]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  20 of 20 completed


In [ ]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,AZO,CHD,DTE,DUK,EVRG,EXC,HSY,ICE,MCD,MDLZ,NEE,NI,O,ORLY,PEG,PPL,SO,SRE,VZ,WM
Date,,,,,,,,,,,,,,,,,,,,
2022-01-03,2045.109985,98.757210,109.680992,93.379715,61.367886,37.294476,183.493134,129.189636,253.779907,62.195683,85.932907,25.153751,62.734241,695.940002,60.583618,27.614265,61.774128,60.977718,44.100864,156.225571
2022-01-04,2080.250000,98.864006,109.865051,93.307533,61.205830,37.255264,185.060120,129.305389,253.713745,62.648220,85.014137,25.245153,63.659382,698.859985,60.427948,27.476837,61.774128,62.077248,44.967079,157.723114
2022-01-05,2034.030029,99.563110,109.745415,93.515068,60.782669,37.229122,187.092529,127.685066,252.466492,63.553307,84.123489,25.382257,62.628521,685.330017,60.281437,27.531809,61.846619,62.456078,45.429619,155.361572
2022-01-06,2062.489990,99.728172,110.039909,93.361664,60.548557,37.072292,186.684128,128.427734,254.828705,63.525017,80.410912,25.272573,62.919270,700.219971,60.336384,27.495159,61.774128,62.904209,45.210957,156.331146
2022-01-07,2014.770020,100.592346,110.150352,94.291054,60.818680,37.013477,186.000351,128.514511,252.343613,63.685291,81.010933,25.473656,62.919270,682.250000,61.352825,27.715048,62.200027,63.851284,46.164200,155.284775
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-23,2430.649902,80.633987,112.328308,96.570061,59.133858,40.695072,228.077362,100.578888,258.487976,64.403381,80.594124,26.183197,59.100235,833.549988,57.814514,27.940247,67.578255,75.044014,34.070160,155.292267
2022-12-27,2454.320068,80.486603,112.982147,97.207680,59.264565,40.799080,227.932297,99.894104,257.782715,64.777710,80.431358,26.268238,58.824970,843.840027,58.505489,28.082123,68.077515,75.562538,34.815250,155.575089
2022-12-28,2461.629883,79.513794,112.025078,96.485680,59.031166,40.600517,225.166901,100.050629,256.111420,64.144241,79.627045,26.032009,58.017506,844.619995,58.013283,27.845661,67.493484,74.363739,34.424969,153.497528


In [ ]:
start=stock_price.loc["2022-01-03"]
end = stock_price.loc["2022-10-12"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2022-10-12"]/OEX.loc["2022-01-03"]-1)

In [ ]:
beta = stock_drop/market_drop
1-beta.T@weight

weight    0.392978
dtype: float64

In [ ]:
stock_drop

Ticker
AZO     0.098225
CHD     0.287328
DTE     0.117157
DUK     0.145598
EVRG    0.170252
EXC     0.096194
HSY     0.177673
ICE     0.316300
MCD     0.102644
MDLZ    0.122992
NEE     0.193472
NI      0.099720
O       0.165241
ORLY    0.045435
PEG     0.171973
PPL     0.185014
SO      0.072015
SRE     0.089635
VZ      0.282358
WM      0.040052
dtype: float64

In [ ]:
print(len(weight))
weight


20


,weight
AZO,0.050462
CHD,0.050462
DTE,0.050462
DUK,0.050462
EVRG,0.050462
EXC,0.050462
HSY,0.041218
ICE,0.050462
MCD,0.050462
MDLZ,0.050462


In [ ]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    #row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    #row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
AZO,0.050462
CHD,0.050462
DTE,0.050462
DUK,0.050462
EVRG,0.050462
EXC,0.050462
HSY,0.041218
ICE,0.050462
MCD,0.050462
MDLZ,0.050462


In [ ]:
a=stock_price@weight
b=1-a.loc["2022-10-12"]/a.loc["2022-01-03"]
1-b/market_drop

weight    1.050055
dtype: float64

In [ ]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [ ]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


In [ ]:
temp=injected_result.loc[["2022-01-03","2022-10-12"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.5488854142960404


,portfolio_value,oex
Date,,
2022-01-03,10000.000000,10000.000000
2022-10-12,8894.931951,7550.360631


In [ ]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
AZO,0.050462,1.261022,1.389165,0.05,0.08905,0.06,0.06
SRE,0.050462,1.003304,5.592761,0.05,0.070851,0.06,0.06
VZ,0.050462,0.981664,2.151405,0.05,0.069322,0.06,0.06
ORLY,0.050462,0.854506,1.133287,0.05,0.060343,0.06,0.06
CHD,0.050462,0.822686,1.057825,0.05,0.058096,0.06,0.06
DUK,0.050462,0.787586,1.215082,0.05,0.055617,0.058795,0.06
EXC,0.050462,0.741579,2.710706,0.05,0.052368,0.055546,0.06
SO,0.050462,0.737209,1.078739,0.05,0.05206,0.055237,0.06
EVRG,0.050462,0.678598,1.226239,0.05,0.047921,0.051098,0.06
NEE,0.050462,0.650217,1.229183,0.05,0.045917,0.049094,0.06


In [ ]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  20 of 20 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  20 of 20 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  20 of 20 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  20 of 20 completed


#